# Markov Decision Processes (MDPs) and Value Iteration

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand Markov Decision Processes (MDPs) fundamentals
- Implement simple MDPs and value iteration algorithms
- See how MDPs form the foundation of reinforcement learning
- Apply value iteration to solve decision-making problems

## 🔗 Where this fits

**Builds on:** Course 02 — Unit 3, lesson 03 "Hidden Markov Models" — the same Markov assumption, now with actions you choose and rewards you collect.

**Used later in:** Course 09 (AIAT 123) — Unit 1, which starts from exactly this MDP and value iteration and builds reinforcement learning on top.

---

This notebook covers practical activities from **Course 02, Unit 3**:
- Introduction to the MDP framework that underlies reinforcement learning (full RL environments and agents come later in the diploma)
- Implementing simple MDPs and value iteration algorithms

---

## Introduction to MDPs

**Markov Decision Processes (MDPs)** are mathematical frameworks for modeling decision-making in situations where outcomes are partly random and partly under control.


## 🌍 The case: the rule in every warehouse in the world

**RAND Corporation, 1950s.** Richard Bellman formulated the Markov decision process and the equation that carries his name (reference 1 below) for sequential decision problems in operations research — inventory, maintenance, resource allocation. Not for games.

**Inventory, 1960.** Herbert Scarf proved that for the standard stochastic inventory problem with a fixed ordering cost, the *optimal* policy has a remarkably simple shape: **when stock falls to or below a level `s`, order back up to a level `S`** — nothing else, ever. That result (*The Optimality of (s, S) Policies in the Dynamic Inventory Problem*, 1960, proved with a tool he invented for it called K-convexity) is why (s, S) rules remain the standard shape of replenishment policies in retail and distribution to this day. Somebody solved the MDP once; the answer became a business rule that buyers apply without knowing where it came from.

**Games and beyond, 2020.** **MuZero** (Schrittwieser *et al.*, Nature 588, 604–609 — reference 2) mastered Go, chess, shogi and Atari by *learning* the transition and reward model and then planning inside it. The framework it plans in is the one on this page.

### What goes wrong without this

Decide greedily and you decide badly, in a way that is invisible at the moment of deciding. A buyer who orders whatever minimises this week's cost stocks out next month. A robot that always takes the cheapest immediate move walks into the dead end. A recommender that always shows the highest-click-probability item trains its users to stop trusting it.

The MDP is the smallest model in which "cheap now, expensive later" can even be *written down*, and value iteration is the proof that a full plan can be **computed from the model**, with no trial and error at all. Watch for one number in the output below: `V*(S2) = 100`, while the immediate reward for being in S2 is only 10. The value of a state is not its reward — it is everything that follows from it. That gap is the entire subject.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import numpy as np

print("✅ Libraries imported!")
print("Ready to work with MDPs and Value Iteration!")


✅ Libraries imported!
Ready to work with MDPs and Value Iteration!


## Part 1: Simple MDP Implementation

Let's create a simple grid world MDP.


In [2]:
# Define a Markov Decision Process: states, actions, stochastic transitions, and rewards.
# Why: an MDP is the standard model for sequential decisions where actions have uncertain outcomes -
# every later reinforcement-learning method in the diploma assumes exactly this structure.

class SimpleMDP:
    """Simple Markov Decision Process implementation"""
    
    def __init__(self, states, actions, transitions, rewards, gamma=0.9):
        """
        Parameters:
        - states: List of states
        - actions: List of actions
        - transitions: Dict[state][action][next_state] = probability
        - rewards: Dict[state][action] = reward
        - gamma: Discount factor
        """
        self.states = states
        self.actions = actions
        self.transitions = transitions
        self.rewards = rewards
        self.gamma = gamma
    
    def get_reward(self, state, action):
        """Get reward for state-action pair"""
        return self.rewards.get(state, {}).get(action, 0.0)
    
    def get_transition_prob(self, state, action, next_state):
        """Get transition probability P(next_state | state, action)"""
        return self.transitions.get(state, {}).get(action, {}).get(next_state, 0.0)

# Example: Simple 3-state MDP
states = ['S0', 'S1', 'S2']
actions = ['Left', 'Right']

# Transition probabilities: P(next_state | current_state, action)
transitions = {
    'S0': {
        'Left': {'S0': 0.8, 'S1': 0.2},
        'Right': {'S1': 0.9, 'S2': 0.1}
    },
    'S1': {
        'Left': {'S0': 0.7, 'S1': 0.3},
        'Right': {'S1': 0.5, 'S2': 0.5}
    },
    'S2': {
        'Left': {'S1': 1.0},
        'Right': {'S2': 1.0}  # Absorbing state: 'Right' in S2 loops back to S2 forever
    }
}

# Rewards: R(state, action)
rewards = {
    'S0': {'Left': -1, 'Right': 0},
    'S1': {'Left': 0, 'Right': 5},
    'S2': {'Left': 0, 'Right': 10}  # S2 is the goal: staying 'Right' collects +10 EVERY step
}

mdp = SimpleMDP(states, actions, transitions, rewards)

print("=" * 60)
print("Simple MDP: Grid World")
print("=" * 60)
print(f"States: {states}")
print(f"Actions: {actions}")
print(f"Discount factor (gamma): {mdp.gamma}")

Simple MDP: Grid World
States: ['S0', 'S1', 'S2']
Actions: ['Left', 'Right']
Discount factor (gamma): 0.9


## Part 2: Value Iteration Algorithm

Value iteration computes the optimal value function V*(s) for all states.

One thing to watch for in the output: S2 is an **absorbing goal state** — the `Right` action loops back to S2 and collects +10 on every step. With discount factor γ = 0.9, its optimal value is the geometric series

$$V^*(S_2) = 10 + 10\gamma + 10\gamma^2 + \dots = \frac{10}{1-\gamma} = 100$$

so a value near 100 (not 10) is exactly right.


In [3]:
# Value iteration: repeatedly apply the Bellman optimality update until values stop changing,
# then read off the optimal policy (best action per state).
# Why: this shows an optimal plan can be COMPUTED from the model - no trial-and-error learning needed.

def value_iteration(mdp, theta=1e-6, max_iterations=500):
    """
    Value iteration algorithm to find optimal value function
    
    Parameters:
    - mdp: MDP object
    - theta: Convergence threshold
    - max_iterations: Maximum iterations
    
    Returns:
    - V: Optimal value function
    - policy: Optimal policy
    """
    # Initialize value function
    V = {state: 0.0 for state in mdp.states}
    
    for iteration in range(max_iterations):
        V_old = V.copy()
        
        # Update value for each state
        for state in mdp.states:
            # Compute Q-value for each action
            Q_values = []
            for action in mdp.actions:
                # Q(s,a) = R(s,a) + gamma * sum(P(s'|s,a) * V(s'))
                q_value = mdp.get_reward(state, action)
                for next_state in mdp.states:
                    prob = mdp.get_transition_prob(state, action, next_state)
                    q_value += mdp.gamma * prob * V_old[next_state]
                Q_values.append(q_value)
            
            # Value is maximum Q-value (optimal action)
            V[state] = max(Q_values) if Q_values else 0.0
        
        # Check convergence
        max_diff = max(abs(V[state] - V_old[state]) for state in mdp.states)
        if max_diff < theta:
            print(f"Converged after {iteration + 1} iterations")
            break
    
    # Extract optimal policy
    policy = {}
    for state in mdp.states:
        Q_values = []
        for action in mdp.actions:
            q_value = mdp.get_reward(state, action)
            for next_state in mdp.states:
                prob = mdp.get_transition_prob(state, action, next_state)
                q_value += mdp.gamma * prob * V[next_state]
            Q_values.append((q_value, action))
        # Choose action with highest Q-value
        policy[state] = max(Q_values, key=lambda x: x[0])[1]
    
    return V, policy

# Run value iteration
print("=" * 60)
print("Running Value Iteration:")
print("=" * 60)

V_star, optimal_policy = value_iteration(mdp)

print("\nOptimal Value Function V*(s):")
for state, value in V_star.items():
    print(f"  V*({state}) = {value:.4f}")

print("\nOptimal Policy π*(s):")
for state, action in optimal_policy.items():
    print(f"  π*({state}) = {action}")

# Why is V*(S2) so large? S2 is absorbing: staying 'Right' pays +10 every step.
gamma = mdp.gamma
print(f"\n💡 Check: V*(S2) = 10 + 10x{gamma} + 10x{gamma}^2 + ... = 10/(1-{gamma}) = {10/(1-gamma):.1f}")
print(f"   Computed V*(S2) = {V_star['S2']:.4f} — matches the geometric series.")

Running Value Iteration:
Converged after 154 iterations

Optimal Value Function V*(s):
  V*(S0) = 82.6364
  V*(S1) = 90.9091
  V*(S2) = 100.0000

Optimal Policy π*(s):
  π*(S0) = Right
  π*(S1) = Right
  π*(S2) = Right

💡 Check: V*(S2) = 10 + 10x0.9 + 10x0.9^2 + ... = 10/(1-0.9) = 100.0
   Computed V*(S2) = 100.0000 — matches the geometric series.


## Part 3: Experiment — change one number, watch the decision change

The run above used **γ = 0.9** and produced V\* = (82.64, 90.91, 100.00), the policy `Right` in every state, converging after 154 sweeps.

Now hold everything else fixed and change **exactly one thing**: the discount factor. γ is not something you measure in the world — it is a modelling choice about how much a reward one step from now is worth compared to a reward now. So it is worth knowing precisely what that choice buys and what it costs.

Two experiments, both re-running the same Bellman update:

1. **Same MDP, four values of γ.** Does the *policy* move, or only the *values*?
2. **One reward changed** — `S0`'s `Left` action now pays **+3 immediately** instead of −1, while `Right` still pays 0 now and leads toward the +10 goal. Sweep γ again and let the code find the exact γ at which the recommended action reverses.


In [4]:
# COMPARATIVE EXPERIMENT: change ONE thing (gamma) and re-run value iteration.
# Why a second implementation? The value_iteration() above prints a running commentary and does
# not report how many sweeps it needed. This one is silent and returns the sweep count, so we can
# put many runs side by side in a table. The Bellman update inside is line-for-line the same.

def solve_quietly(reward_table, gamma, theta=1e-6, max_iterations=500):
    """Value iteration, silent. Returns (V, policy, sweeps); sweeps is None if it never converged."""
    V = {s: 0.0 for s in states}
    sweeps = None
    for i in range(max_iterations):
        V_old = V.copy()
        for s in states:
            # Q(s,a) = R(s,a) + gamma * sum_s' P(s'|s,a) V(s')  -- the same update as above
            V[s] = max(reward_table[s][a] + gamma * sum(transitions[s][a].get(ns, 0.0) * V_old[ns]
                                                        for ns in states)
                       for a in actions)
        if max(abs(V[s] - V_old[s]) for s in states) < theta:
            sweeps = i + 1          # converged: record how many sweeps it took
            break
    # Read the greedy policy off the converged values, exactly as value_iteration() does.
    policy = {s: max(actions,
                     key=lambda a: reward_table[s][a] + gamma * sum(transitions[s][a].get(ns, 0.0) * V[ns]
                                                                   for ns in states))
              for s in states}
    return V, policy, sweeps


print("=" * 78)
print("EXPERIMENT 1 - same MDP, same rewards, four discount factors")
print("=" * 78)
print(f"{'gamma':>6} {'sweeps':>8}  {'V*(S0)':>9} {'V*(S1)':>9} {'V*(S2)':>9}   policy (S0, S1, S2)")
runs = {}
for g in [0.0, 0.5, 0.9, 0.99]:
    V, pol, sweeps = solve_quietly(rewards, g)
    runs[g] = (V, pol, sweeps)
    shown = sweeps if sweeps is not None else "500+"   # 500 is the iteration cap
    print(f"{g:>6} {shown:>8}  {V['S0']:>9.2f} {V['S1']:>9.2f} {V['S2']:>9.2f}   "
          f"{pol['S0']}, {pol['S1']}, {pol['S2']}")

# Conclusions are DERIVED from the runs above, never hard-coded.
policies = {tuple(runs[g][1][s] for s in states) for g in runs}
v_low, v_high = runs[0.0][0]['S2'], runs[0.99][0]['S2']
print()
print("What these numbers actually support:")
print(f"   - The optimal POLICY is identical at every gamma tested "
      f"({'unchanged' if len(policies) == 1 else 'it changed'}): "
      + ", ".join(runs[0.9][1][s] for s in states))
print(f"   - The VALUES are not: V*(S2) runs from {v_low:.2f} at gamma=0 to {v_high:.2f} at gamma=0.99.")
print(f"   - The COST of solving is not either: {runs[0.0][2]} sweeps at gamma=0, "
      f"{runs[0.9][2]} at gamma=0.9, and at gamma=0.99 the loop hit its 500-sweep cap")
print( "     WITHOUT reaching the 1e-6 threshold - it returned a number anyway, unconverged.")
print()

print("=" * 78)
print("EXPERIMENT 2 - change ONE reward, then sweep gamma again")
print("=" * 78)
print("Only S0's 'Left' reward changes: -1  ->  +3.")
print("'Left' now pays 3 immediately but mostly loops back to S0 (p=0.8).")
print("'Right' still pays 0 now, but moves toward S1/S2 where the +10 goal lives.")
print()

# Copy the reward table and change exactly one entry.
rewards_myopic = {s: dict(a_r) for s, a_r in rewards.items()}
rewards_myopic['S0']['Left'] = 3

print(f"{'gamma':>6}   best action in S0   {'V*(S0)':>9}")
for g in [0.0, 0.3, 0.5, 0.6, 0.9]:
    V, pol, _ = solve_quietly(rewards_myopic, g)
    print(f"{g:>6}   {pol['S0']:<17} {V['S0']:>9.2f}")

# Let the code find the crossover instead of asserting it: bisect on gamma.
lo, hi = 0.0, 0.9
for _ in range(40):
    mid = (lo + hi) / 2
    if solve_quietly(rewards_myopic, mid)[1]['S0'] == 'Left':
        lo = mid          # still myopic: grab the +3
    else:
        hi = mid          # patient enough to invest
print()
print(f"   Crossover found by bisection: gamma ~ {lo:.3f}")
print( "   Below it, the optimal action in S0 is Left  - take the 3 now.")
print( "   Above it, the optimal action in S0 is Right - give up the 3 to reach the +10 sooner.")
print()
print( "   Nothing about the world changed between those two rows. One number in the MODEL")
print( "   changed, and the recommended decision reversed. gamma is a value judgement")
print( "   wearing a Greek letter, and somebody on your team has to own it.")


EXPERIMENT 1 - same MDP, same rewards, four discount factors
 gamma   sweeps     V*(S0)    V*(S1)    V*(S2)   policy (S0, S1, S2)
   0.0        2       0.00      5.00     10.00   Right, Right, Right
   0.5       25       7.00     13.33     20.00   Right, Right, Right
   0.9      154      82.64     90.91    100.00   Right, Right, Right
  0.99     500+     974.61    983.53    993.43   Right, Right, Right

What these numbers actually support:
   - The optimal POLICY is identical at every gamma tested (unchanged): Right, Right, Right
   - The VALUES are not: V*(S2) runs from 10.00 at gamma=0 to 993.43 at gamma=0.99.
   - The COST of solving is not either: 2 sweeps at gamma=0, 154 at gamma=0.9, and at gamma=0.99 the loop hit its 500-sweep cap
     WITHOUT reaching the 1e-6 threshold - it returned a number anyway, unconverged.

EXPERIMENT 2 - change ONE reward, then sweep gamma again
Only S0's 'Left' reward changes: -1  ->  +3.
'Left' now pays 3 immediately but mostly loops back to S0 (p=0.8

## 💬 Discuss

The two experiments above are the evidence. None of these has a settled answer.

1. **Who chooses γ, and how would you defend the choice to a client?** In Experiment 2 the optimal action reverses at γ ≈ 0.518. Below it the model says take the money now; above it, invest. For a Saudi retailer deciding how much stock to hold before Ramadan, or a maintenance team deciding whether to replace a part now or run it another quarter, what would you set γ to — and what evidence would you bring? Is γ a finance question, an engineering question, or an ethics question when the "reward" is a patient outcome?
2. **In Experiment 1 the policy never moved while the values moved by two orders of magnitude.** A colleague concludes: "the values do not matter, only the policy does — stop worrying about γ." Argue against them. Name a situation where the *values* are the deliverable, not the policy.
3. **At γ = 0.99, value iteration hit its 500-sweep cap and returned numbers anyway, with no error.** The policy read off those unconverged values happened to be right. Would you ship code that behaves this way? What would you change — raise the cap, loosen the threshold, return a convergence flag, refuse to return at all? Each choice moves a risk somewhere; say where.

---


## ⚠️ Where this breaks

Value iteration is exact, terminates, and comes with strong guarantees — and every one of those guarantees rests on an assumption you should be able to state out loud.

- **You must already know P(s′ | s, a) and R(s, a).** Value iteration is *planning*, not learning: it computes an optimal policy from a model that somebody handed it. In this notebook, both tables were typed into the cell. When you do not have them — which is most of the time — you need reinforcement learning, which is exactly where Course 09 picks this up. Everything here is the answer to "what should I do *if the model is right*".
- **γ is a modelling choice, and Experiment 2 shows it choosing the decision.** At γ ≈ 0.518 the optimal action reverses. It is not measured, it is not learned, and it is not neutral. Discounting a patient's future health at 0.9 per step is an ethical statement, not a hyperparameter.
- **γ close to 1 makes the arithmetic slow and the values huge.** Convergence of value iteration is governed by γ: 2 sweeps at γ = 0, 154 at 0.9, and at 0.99 the loop exhausted its 500-sweep cap without reaching the 1e-6 threshold. For infinite-horizon problems where you genuinely do not want to discount, you need average-reward formulations or policy iteration instead.
- **The state space is the wall — the curse of dimensionality.** Value iteration sweeps *every* state on *every* iteration. Three states is trivial. A warehouse with 20 products at 100 stock levels each has 100²⁰ states, and no amount of engineering will enumerate them. Real systems use function approximation, factored representations or sampling; that is the whole reason deep reinforcement learning exists.
- **The Markov assumption must actually hold.** "The next state depends only on the current state and action" is a strong claim about your state encoding. If the right decision depends on something not in the state — how long the machine has been running, what the customer bought last month — the model is wrong, and value iteration will return a confidently optimal policy for the wrong problem.
- **The reward function is written by a human, and the agent will exploit exactly what you wrote.** `S2` pays +10 every step forever, so the optimal policy is to sit in S2 and collect — which is right here because S2 *is* the goal. Change the numbers carelessly and you get an agent that games the reward instead of doing the job. Reward specification is the hard part of every applied MDP.
- **The cheaper alternative.** If the horizon is short and the branching small, plain expected-utility calculation (Unit 3, notebook 01) or a decision tree drawn on a whiteboard answers the question without any of this machinery. And if the MDP has already been solved *in general* for your problem class — as Scarf did for inventory in 1960 — the professional move is to look up the known optimal policy shape and fit its two parameters, not to re-derive it.

---


## Summary

### Key Concepts:
1. **MDP Components**: States, actions, transition probabilities, rewards, discount factor
2. **Value Function V(s)**: Expected cumulative reward from state s
3. **Value Iteration**: Algorithm to compute optimal value function
4. **Policy**: Mapping from states to actions

### Applications:
- Robotics (path planning)
- Game AI
- Resource allocation
- Autonomous systems

**Reference:** Course 02, Unit 3: "Implementing simple MDPs and value iteration algorithms"


## 📚 References

1. Bellman, R. (1957). *A Markovian Decision Process*. Journal of Mathematics and Mechanics 6(5), 679-684. (the MDP and the Bellman equation)
2. Schrittwieser, J., et al. (2020). *Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model*. Nature 588, 604-609. <https://arxiv.org/abs/1911.08265> (MuZero: planning in a learned MDP)
3. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.), Chapters 3-4 (Finite MDPs; Dynamic Programming). MIT Press. <http://incompleteideas.net/book/the-book-2nd.html>